# Climate Crop Yield Intelligence 🌾🌍
### From warming to yield: where climate pressure hits agriculture first

This is a **portfolio case study**, not a claim about a real client. I use real public data for six crops from **1990–2023**.

My question is simple:
> **Which crop-country histories deserve attention first, and how much prediction value does climate add beyond simple historical baselines?**

I explain important choices as: **I found X → I changed Y → because Z.** The climate results are observational signals, not causal proof.

## 1. Importing Libraries & Dataset
I download public OWID Grapher CSVs directly. In Colab, OWID can reject Python's default downloader with HTTP 403, so I use a normal request header. No manual dataset upload is needed.

In [ ]:
from pathlib import Path
import sys,io,json,warnings,requests,numpy as np,pandas as pd,matplotlib.pyplot as plt,seaborn as sns
from IPython.display import display,Markdown
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error,r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid"); pd.set_option("display.max_columns",40)
IS_COLAB = "google.colab" in sys.modules
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
FIG_DIR = Path("/content/climate_crop_figures") if IS_COLAB else ROOT / "reports" / "figures"
TABLE_DIR = Path("/content/climate_crop_tables") if IS_COLAB else ROOT / "reports" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
BASE="https://ourworldindata.org/grapher"; START,END=1990,2023
CROPS={"Wheat":"wheat-yields","Maize":"maize-yields","Rice":"rice-yields","Potatoes":"potato-yields","Soybeans":"soybean-yields","Barley":"barley-yields"}
DRIVERS={"temperature_c":"average-annual-surface-temperature","precipitation_mm":"average-precipitation-per-year","fertilizer_kg_ha":"fertilizer-use-in-kg-per-hectare-of-arable-land","irrigated_land_pct":"agricultural-land-irrigation"}
def read_series(slug,name):
    u=f"{BASE}/{slug}.csv?v=1&csvType=full&useColumnShortNames=false"
    r=requests.get(u,headers={"User-Agent":"Mozilla/5.0 (compatible; ClimateCropYieldIntelligence/1.0)","Referer":"https://ourworldindata.org/"},timeout=60); r.raise_for_status()
    x=pd.read_csv(io.StringIO(r.text)); vals=[z for z in x.columns if z not in {"Entity","Code","Year"}]
    if len(vals)!=1: raise ValueError(f"Unexpected OWID schema for {slug}: {vals}")
    x=x.rename(columns={vals[0]:name})[["Entity","Code","Year",name]]; x=x[x.Code.astype("string").str.fullmatch(r"[A-Z]{3}",na=False)].copy(); x["Year"]=pd.to_numeric(x.Year,errors="coerce"); x=x.dropna(subset=["Year"]); x["Year"]=x.Year.astype(int); return x
y=[]
for crop,slug in CROPS.items():
    q=read_series(slug,"yield_t_ha"); q["crop"]=crop; y.append(q)
y=pd.concat(y,ignore_index=True)
d=[read_series(slug,name) for name,slug in DRIVERS.items()]; drivers=d[0]
for q in d[1:]:
    v=[z for z in q.columns if z not in {"Entity","Code","Year"}]; drivers=drivers.merge(q[["Code","Year"]+v],on=["Code","Year"],how="outer",validate="one_to_one")
df=y.merge(drivers.drop(columns="Entity"),on=["Code","Year"],how="left",validate="many_to_one"); df=df[df.Year.between(START,END)].drop_duplicates(["Code","Year","crop"]).sort_values(["crop","Code","Year"]).reset_index(drop=True)
print(f"Rows: {len(df):,} | Countries/territories: {df.Code.nunique()} | Crops: {df.crop.nunique()} | Years: {df.Year.min()}–{df.Year.max()}")
display(pd.DataFrame({"coverage_pct":(100*df.notna().mean()).round(2)}).sort_values("coverage_pct"))

### What I found → what I decided
Irrigation has much lower coverage than the other variables, so I keep it **exploratory only**. I also found that crop yield scales are very different, so I do **not** rank cross-crop climate sensitivity using raw `t/ha` slopes.

## 2. Preparing the analysis
For EDA, I compare each year with the same country's climate normal. For temperature sensitivity, I remove the linear time trend inside each country × crop history. Then I express yield residuals as **% of that system's mean yield** so crops are more comparable.

In [ ]:
cl=df[["Code","Year","temperature_c","precipitation_mm"]].drop_duplicates(["Code","Year"]); n=cl.groupby("Code")[["temperature_c","precipitation_mm"]].mean()
df["temp_deviation_c"]=df.temperature_c-df.Code.map(n.temperature_c); df["precip_deviation_mm"]=df.precipitation_mm-df.Code.map(n.precipitation_mm)
o=df.sort_values(["Code","crop","Year"]); df["yield_yoy_pct"]=o.groupby(["Code","crop"]).yield_t_ha.pct_change(fill_method=None).mul(100).reindex(df.index); lo,hi=df.yield_yoy_pct.quantile([.01,.99]); df["yield_yoy_pct_w"]=df.yield_yoy_pct.clip(lo,hi)
def resid(year,val):
    co=np.polyfit(year.astype(float),val.astype(float),1); return val.astype(float)-np.polyval(co,year.astype(float))
def detrend(data,min_n=8):
    out=data.copy(); out["temp_detrended_c"]=np.nan; out["yield_detrended_t_ha"]=np.nan; out["yield_detrended_pct"]=np.nan
    for _,p in out.groupby(["Code","crop"]):
        v=p.dropna(subset=["Year","temperature_c","yield_t_ha"]);
        if len(v)<min_n or v.Year.nunique()<min_n: continue
        yr=v.Year.to_numpy(float); yy=v.yield_t_ha.to_numpy(float); mu=float(yy.mean())
        if mu<=0: continue
        tr=resid(yr,v.temperature_c.to_numpy(float)); yrx=resid(yr,yy); out.loc[v.index,"temp_detrended_c"]=tr; out.loc[v.index,"yield_detrended_t_ha"]=yrx; out.loc[v.index,"yield_detrended_pct"]=100*yrx/mu
    return out
det=detrend(df)

## 3. Long-run yield change — matched countries
I found that comparing all countries in the early window with all countries in the recent window can change the sample. So I match the **same country-crop histories** in both windows and require at least 3 observations in each.

In [ ]:
def yield_change(data):
    a=data[data.Year.between(1990,1994)].groupby(["Code","crop"]).yield_t_ha.agg(early="median",n1="count").reset_index(); b=data[data.Year.between(2019,2023)].groupby(["Code","crop"]).yield_t_ha.agg(recent="median",n2="count").reset_index(); z=a.merge(b,on=["Code","crop"]); z=z[(z.n1>=3)&(z.n2>=3)&(z.early>0)]
    rows=[]
    for crop,p in z.groupby("crop"):
        e=float(p.early.median()); r=float(p.recent.median()); rows.append({"crop":crop,"early_median":e,"recent_median":r,"change_pct":100*(r-e)/e,"matched_countries":p.Code.nunique()})
    return pd.DataFrame(rows).sort_values("change_pct",ascending=False).reset_index(drop=True)
yc=yield_change(df); display(yc.round(2))
plt.figure(figsize=(9,5)); ax=sns.barplot(data=yc,x="crop",y="change_pct",hue="crop",palette="Set2",legend=False); plt.title("Matched-country Median Yield Change"); plt.ylabel("1990–1994 to 2019–2023 change (%)"); plt.xlabel("")
for p,v in zip(ax.patches,yc.change_pct): ax.annotate(f"{v:.1f}%",(p.get_x()+p.get_width()/2,p.get_height()),ha="center",va="bottom",xytext=(0,3),textcoords="offset points")
plt.tight_layout(); plt.savefig(FIG_DIR/"01_matched_yield_change.png",dpi=180,bbox_inches="tight"); plt.show()

## 4. Relative temperature sensitivity
A raw `t/ha per +1°C` comparison can make high-yield crops look mechanically more sensitive. I use **% yield deviation per +1°C** as the main cross-crop measure. Negative means warmer-than-trend years tend to align with yield below that system's own trend. This is still association, not causation.

In [ ]:
v=det.dropna(subset=["temp_detrended_c","yield_detrended_pct","yield_detrended_t_ha"]); rows=[]
for crop,p in v.groupby("crop"):
    x=p.temp_detrended_c.to_numpy(float)
    if len(p)>=20 and np.std(x)>=1e-8: rows.append({"crop":crop,"yield_change_pct_per_1c":np.polyfit(x,p.yield_detrended_pct.to_numpy(float),1)[0],"yield_change_t_ha_per_1c":np.polyfit(x,p.yield_detrended_t_ha.to_numpy(float),1)[0]})
sens=pd.DataFrame(rows).sort_values("yield_change_pct_per_1c").reset_index(drop=True); display(sens.round(4))
plt.figure(figsize=(9,5)); ax=sns.barplot(data=sens,x="crop",y="yield_change_pct_per_1c",hue="crop",palette="Set2",legend=False); plt.axhline(0,color="black",lw=1); plt.title("Detrended Yield Association per +1°C"); plt.ylabel("Yield deviation (% of system mean per +1°C)"); plt.xlabel("")
for p,val in zip(ax.patches,sens.yield_change_pct_per_1c): ax.annotate(f"{val:.2f}%",(p.get_x()+p.get_width()/2,p.get_height()),ha="center",va="top" if val<0 else "bottom",xytext=(0,-4 if val<0 else 3),textcoords="offset points")
plt.tight_layout(); plt.savefig(FIG_DIR/"03_temperature_sensitivity_relative.png",dpi=180,bbox_inches="tight"); plt.show()

## 5. Temperature ranges & management
I do not choose one exact degree and call it the optimum. I build **8 temperature quantile groups inside each crop**. For irrigation/fertilizer plots I normalize yield to each crop's median and create quartiles inside each crop, so absolute crop scale does not dominate the visual.

In [ ]:
tp=[]
for crop,p in df.dropna(subset=["temperature_c","yield_t_ha"]).groupby("crop"):
    q=p.copy(); q["temp_bin"]=pd.qcut(q.temperature_c,8,duplicates="drop"); s=q.groupby("temp_bin",observed=True).yield_t_ha.median().reset_index(); s["crop"]=crop; s["temp_bin"]=s.temp_bin.astype(str); tp.append(s)
ts=pd.concat(tp,ignore_index=True); g=sns.catplot(data=ts,x="temp_bin",y="yield_t_ha",col="crop",col_wrap=2,kind="bar",palette="Greens",sharex=False,sharey=False,height=3.2,aspect=1.35); g.set_xticklabels(rotation=55,ha="right"); g.set_axis_labels("Within-crop temperature range","Median yield (t/ha)"); g.fig.suptitle("Temperature Ranges — Exploratory, Not a Magic Degree",y=1.02); plt.show()
df["yield_index_pct"]=100*df.yield_t_ha/df.groupby("crop").yield_t_ha.transform("median")
fig,axes=plt.subplots(1,2,figsize=(14,5))
for ax,col,title in [(axes[0],"irrigated_land_pct","Irrigation"),(axes[1],"fertilizer_kg_ha","Fertilizer")]:
    parts=[]
    for crop,p in df.dropna(subset=[col,"yield_index_pct"]).groupby("crop"):
        if p[col].nunique()<4: continue
        q=p.copy(); q["group"]=pd.qcut(q[col],4,labels=["Low","Mid-low","Mid-high","High"],duplicates="drop"); parts.append(q)
    q=pd.concat(parts,ignore_index=True); sns.boxplot(data=q,x="group",y="yield_index_pct",hue="group",palette="Set2",legend=False,showfliers=False,ax=ax); ax.axhline(100,color="black",ls="--",lw=1); ax.set_title(f"Crop-normalized yield by {title} group — descriptive"); ax.set_ylabel("Yield index (% of crop median)")
plt.tight_layout(); plt.show()

## 6. Priority screening
I call this a **screening score**, not a risk probability. It combines detrended yield volatility (%) and the negative relative temperature slope (% yield per +1°C). I require at least **20 usable observations** before a country-crop history can enter the ranking.

In [ ]:
ds=detrend(df,20); base=ds.groupby(["Code","Entity","crop"]).agg(detrended_yield_std_pct=("yield_detrended_pct","std"),observations=("yield_detrended_pct","count")).reset_index(); vv=ds.dropna(subset=["temp_detrended_c","yield_detrended_pct"]); sl=[]
for (code,crop),p in vv.groupby(["Code","crop"]):
    if len(p)>=20 and p.temp_detrended_c.std()>=1e-8: sl.append({"Code":code,"crop":crop,"temp_slope_pct_per_c":np.polyfit(p.temp_detrended_c.to_numpy(float),p.yield_detrended_pct.to_numpy(float),1)[0]})
screen=base.merge(pd.DataFrame(sl),on=["Code","crop"]); screen=screen[(screen.observations>=20)&screen.detrended_yield_std_pct.notna()].copy(); screen["warming_penalty_pct_per_c"]=(-screen.temp_slope_pct_per_c).clip(lower=0); screen["volatility_rank"]=screen.detrended_yield_std_pct.rank(pct=True); screen["warming_penalty_rank"]=screen.warming_penalty_pct_per_c.rank(pct=True); screen["screening_score"]=(screen.volatility_rank+screen.warming_penalty_rank)/2; screen=screen.sort_values(["screening_score","warming_penalty_pct_per_c"],ascending=[False,False]).reset_index(drop=True)
display(screen.head(10)[["Entity","crop","screening_score","detrended_yield_std_pct","temp_slope_pct_per_c","observations"]].round(4))
plt.figure(figsize=(10,6)); sns.scatterplot(data=screen,x="warming_penalty_pct_per_c",y="detrended_yield_std_pct",hue="crop",size="screening_score",sizes=(25,180),palette="Set2",alpha=.65)
for _,r in screen.head(8).iterrows(): plt.annotate(f"{r.Entity} – {r.crop}",(r.warming_penalty_pct_per_c,r.detrended_yield_std_pct),xytext=(5,5),textcoords="offset points",fontsize=8)
plt.title("Climate Screening — Where Should I Investigate First?"); plt.xlabel("Negative detrended temperature association (% yield per +1°C)"); plt.ylabel("Detrended yield volatility (%)"); plt.tight_layout(); plt.savefig(FIG_DIR/"07_screening_priority.png",dpi=180,bbox_inches="tight"); plt.show()

## 7. Prediction: climate first, fertilizer second
The earlier draft mixed fertilizer into a “climate model”. I changed that. I now run **climate only** first, then add fertilizer. I train on years before 2018 and test on 2018–2023. All anomaly normals are calculated from **training years only** to avoid leakage. I compare against crop median, country × crop median, and persistence.

In [ ]:
SPLIT=2018; RS=42; SRC=["temperature_c","precipitation_mm","fertilizer_kg_ha"]; CLIM=["temp_anomaly_c","precip_anomaly_mm"]; FULL=CLIM+["fertilizer_anomaly_kg_ha"]
train=df[df.Year<SPLIT].copy(); test=df[df.Year>=SPLIT].copy(); glob=float(train.yield_t_ha.median()); cm=train.groupby("crop").yield_t_ha.median(); crop_test=test.crop.map(cm).fillna(glob).to_numpy(); crop_train=train.crop.map(cm).fillna(glob).to_numpy(); cc=train.groupby(["Code","crop"]).yield_t_ha.median()
def mapcc(f,s): return s.reindex(pd.MultiIndex.from_frame(f[["Code","crop"]])).to_numpy(dtype=float)
cc_test=mapcc(test,cc); cc_test=np.where(np.isnan(cc_test),crop_test,cc_test); cc_train=mapcc(train,cc); cc_train=np.where(np.isnan(cc_train),crop_train,cc_train)
last=train.sort_values(["Code","crop","Year"]).groupby(["Code","crop"],as_index=False).tail(1).set_index(["Code","crop"]).yield_t_ha; persistence=mapcc(test,last); persistence=np.where(np.isnan(persistence),cc_test,persistence)
ut=train[["Code","Year"]+SRC].drop_duplicates(["Code","Year"]); normals=ut.groupby("Code")[SRC].mean(); gn=ut[SRC].mean(); names={"temperature_c":"temp_anomaly_c","precipitation_mm":"precip_anomaly_mm","fertilizer_kg_ha":"fertilizer_anomaly_kg_ha"}
for s,t in names.items(): train[t]=train[s]-train.Code.map(normals[s]).fillna(gn[s]); test[t]=test[s]-test.Code.map(normals[s]).fillna(gn[s])
def model(features):
    pre=ColumnTransformer([("num",Pipeline([("imp",SimpleImputer(strategy="median",add_indicator=True))]),features),("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output=True))]),["crop"])]); return Pipeline([("prep",pre),("model",RandomForestRegressor(n_estimators=250,min_samples_leaf=5,random_state=RS,n_jobs=-1))])
def fit(features):
    mm=model(features); cols=features+["crop"]; mm.fit(train[cols],train.yield_t_ha.to_numpy()-cc_train); return mm,cc_test+mm.predict(test[cols])
mc,pc=fit(CLIM); mf,pf=fit(FULL); yt=test.yield_t_ha.to_numpy(); metrics={"Crop median":mean_absolute_error(yt,crop_test),"Country × crop median":mean_absolute_error(yt,cc_test),"Climate only":mean_absolute_error(yt,pc),"Climate + fertilizer":mean_absolute_error(yt,pf),"Persistence":mean_absolute_error(yt,persistence)}; full_r2=r2_score(yt,pf); met=pd.Series(metrics,name="MAE_t_ha").to_frame().sort_values("MAE_t_ha"); display(met.round(4)); print(f"Climate-only improvement vs country × crop median: {100*(metrics['Country × crop median']-metrics['Climate only'])/metrics['Country × crop median']:.2f}%"); print(f"Fertilizer incremental improvement: {100*(metrics['Climate only']-metrics['Climate + fertilizer'])/metrics['Climate only']:.2f}%"); print(f"Full model R²: {full_r2:.4f}")
plot=met.reset_index(); plot.columns=["Method","MAE"]; plt.figure(figsize=(10,5)); ax=sns.barplot(data=plot,x="Method",y="MAE",hue="Method",palette="Set2",legend=False); plt.title("2018+ Holdout — Climate Value, Fertilizer Value and Strong Baselines"); plt.ylabel("MAE (t/ha) — lower is better"); plt.xlabel(""); plt.xticks(rotation=12); plt.tight_layout(); plt.savefig(FIG_DIR/"08_model_ablation_baselines.png",dpi=180,bbox_inches="tight"); plt.show()

### Why these model settings?
I use **250 trees**, `min_samples_leaf=5`, and `random_state=42`. I want a stable, reproducible baseline without turning this small portfolio project into a hyperparameter-search exercise. I do **not** tune on the 2018+ holdout.

## 8. Error analysis & final message
I keep model failure visible. A complex model is not automatically better. If persistence wins, that is a real result.

In [ ]:
pred=test[["Entity","Code","Year","crop","yield_t_ha"]].copy(); pred["predicted_yield_t_ha"]=pf; pred["abs_error"]=np.abs(pred.yield_t_ha-pf); err=pred.groupby("crop").abs_error.agg(["mean","median","count"]).sort_values("mean",ascending=False); display(err.round(4))
best=yc.iloc[0]; strong=sens.iloc[0]; top=screen.iloc[0]; ci=100*(metrics["Country × crop median"]-metrics["Climate only"])/metrics["Country × crop median"]; fi=100*(metrics["Climate only"]-metrics["Climate + fertilizer"])/metrics["Climate only"]
display(Markdown(f"""### What I found
- **{best.crop}** has the largest matched-country median yield increase: **{best.change_pct:.2f}%**.
- The strongest relative detrended temperature association is **{strong.crop}: {strong.yield_change_pct_per_1c:.2f}% per +1°C**.
- The top screening segment is **{top.Entity} – {top.crop}**.
- Climate-only MAE is **{metrics['Climate only']:.4f} t/ha**, improving on the static country × crop median by **{ci:.2f}%**.
- Adding fertilizer improves MAE by another **{fi:.2f}%**, to **{metrics['Climate + fertilizer']:.4f} t/ha**.
- Persistence is still stronger at **{metrics['Persistence']:.4f} MAE**.

**Main message:** climate adds measurable information, fertilizer adds a small extra improvement, but recent production history remains the strongest short-horizon predictor in V1.
"""))

## 9. Limitations & audit notes
This is country-year data. It does not directly capture growing-season heat extremes, rainfall timing, crop calendars, local soils, cultivar choice or farm-level management. The screening score is not a crop-loss probability.

After the final audit I changed six things: **matched country cohorts, relative climate sensitivity, within-crop temperature bins, crop-normalized management plots, climate-only vs climate+fertilizer ablation, and stronger wording around the screening score.**

The reusable rule is: **I found X → I changed Y → because Z → I did not use shortcut A because B.**

In [ ]:
nb_summary={"dataset":{"rows":int(len(df)),"countries":int(df.Code.nunique()),"crops":int(df.crop.nunique()),"start_year":int(df.Year.min()),"end_year":int(df.Year.max())},"matched_country_yield_change":{r.crop:round(float(r.change_pct),2) for _,r in yc.iterrows()},"relative_temperature_sensitivity_pct_per_1c":{r.crop:round(float(r.yield_change_pct_per_1c),4) for _,r in sens.iterrows()},"top_screening_segment":{"country":str(screen.iloc[0].Entity),"crop":str(screen.iloc[0].crop),"screening_score":round(float(screen.iloc[0].screening_score),4)},"model_metrics":{"crop_median_mae":round(float(metrics["Crop median"]),4),"country_crop_median_mae":round(float(metrics["Country × crop median"]),4),"climate_only_mae":round(float(metrics["Climate only"]),4),"climate_plus_fertilizer_mae":round(float(metrics["Climate + fertilizer"]),4),"persistence_mae":round(float(metrics["Persistence"]),4),"full_model_r2":round(float(full_r2),4)}}
(TABLE_DIR/"notebook_summary.json").write_text(json.dumps(nb_summary,indent=2,sort_keys=True)+"\n",encoding="utf-8"); print(f"Notebook summary: {TABLE_DIR/'notebook_summary.json'}"); print(f"Figures: {FIG_DIR}")